# 🤖 Nyaya-Jyoti: AI-Powered Loan Agreement Generator

Welcome to **Nyaya-Jyoti**, an intelligent chatbot-based legal document generator that helps you **create a personalized Loan Agreement** in minutes — without any legal expertise.

This tool is powered by **GPT-Neo** and **Semantic Matching AI** to automatically generate standard as well as special legal clauses. The document is built dynamically based on your answers.

---

### 📄 How it works:

1. **Upload CSV Clause Registry and Loan Agreement Template (.docx)**
2. **Answer simple questions** about the lender, borrower, loan terms, interest, and other legal details
3. **Optionally enter a special clause** (e.g., "part payment allowed") — AI will auto-generate a matching clause
4. **Watch the document update live**, line by line
5. **Download your completed Loan Agreement** as a .docx file

---

### ⚠️ Instructions:

- Make sure to upload:
  - ✅ CSV file containing clause prompts
  - ✅ Word (.docx) template with placeholders like `[Borrower_Name]`, `[Loan_Amount]`, etc.
- Respond to each input **carefully and in the correct format** (e.g., dates in `DD/MM/YYYY`, amounts in numbers/words).
- Special clauses will be handled **after all other fields are filled**.
- The document will download automatically once complete.

---

> 🛡️ This tool is built for educational and prototype purposes under the *Nyaya-Jyoti* project at Bennett University. It is not a substitute for legal advice.



🤝 Let's Begin Your Loan Agreement Creation

📋 Interactive Chat – Fill Your Loan Agreement Step-by-Step

You are now entering the main chatbot flow. Kindly answer the questions one by one.
Each response will automatically update your legal agreement in real-time.

✅ Please ensure:
- ✍️ Names are typed correctly (case-sensitive)
- 📆 Dates follow the format: DD/MM/YYYY
- 💰 Amounts are clearly written in both numbers and words
- 🧾 Special clause, if any, should be a proper legal sentence (e.g., 'final payment via NEFT')

Your responses will be used to generate a formal .docx loan agreement.
----------------------------------------------------------------------


In [ ]:
# @title
# 🛠️ Install required packages
!pip install -q transformers torch pandas sentence-transformers python-docx

# 📦 Imports
import torch
import pandas as pd
import re
import docx
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
from google.colab import files
from IPython.display import display, Markdown

# --- Load Clause Prompt Registry ---
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
prompt_df = pd.read_csv(csv_filename)
prompt_df["parameters"] = prompt_df["parameters"].fillna("").astype(str)
prompt_df["parameters"] = prompt_df["parameters"].apply(
    lambda x: ", ".join(sorted(set(p.strip() for p in x.split(",") if p.strip().lower() != "nan")))
)

# --- Load GPT-Neo & Embedding Model ---
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
instruction_embeddings = embedder.encode(prompt_df['instruction'].tolist(), convert_to_tensor=True)

# --- Clause Generation Utilities ---
def build_prompt(example_1, example_2, instruction):
    return (
        "You are a legal assistant specialized in drafting formal legal clauses.\n"
        f"Example 1:\nClause: {example_1}\nEndClause\n\n"
        f"Example 2:\nClause: {example_2}\nEndClause\n\n"
        f"Now, generate ONLY the legal clause for {instruction}, using formal legal language.\n"
        "Output only the text between the markers 'Clause:' and 'EndClause'.\n\nClause: "
    )

def generate_clause(prompt_text):
    input_ids = tokenizer.encode(prompt_text, return_tensors="pt").to(device)
    output_ids = model.generate(
        input_ids,
        max_length=300,
        temperature=0.35,
        top_k=50,
        top_p=0.85,
        repetition_penalty=1.2,
        do_sample=True,
        num_return_sequences=1,
        pad_token_id=tokenizer.pad_token_id
    )
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if "EndClause" in generated_text:
        return generated_text.split("Clause:")[1].split("EndClause")[0].strip()
    return generated_text.split("Clause:")[1].strip()

def fill_parameters_dynamic(clause_text, param_string):
    placeholders = set(re.findall(r"{(.*?)}", clause_text))
    defined_params = [p.strip() for p in str(param_string).split(',') if p.strip()]
    combined_params = sorted(placeholders.union(set(defined_params)))
    param_values = {}
    for param in combined_params:
        value = input(f"🧾 Please provide value for '{param}': ").strip()
        param_values[param] = value
    for param, value in param_values.items():
        clause_text = clause_text.replace(f"{{{param}}}", value)
    return clause_text, param_values

def find_best_match_semantic(user_input):
    user_embedding = embedder.encode(user_input, convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(user_embedding, instruction_embeddings)[0]
    best_idx = torch.argmax(cosine_scores).item()
    best_score = cosine_scores[best_idx].item()
    if best_score > 0.4:
        return prompt_df.iloc[best_idx]
    return None

# --- Upload Template ---
print("📂 Upload your Loan Agreement DOCX template:")
uploaded_template = files.upload()
template_filename = list(uploaded_template.keys())[0]
doc = docx.Document(template_filename)

# --- Placeholder Explanations ---
explanations = {
    "Borrower_Name": "Enter the full name of the person receiving the loan",
    "Lender_Name": "Enter the full name of the person giving the loan",
    "Location": "Enter the city or place where the agreement is signed",
    "Day": "Enter the numeric day of the month",
    "Month": "Enter the numeric month (e.g., 04 for April)",
    "Year": "Enter the four-digit year",
    "Loan_Amount": "Enter the loan amount in numbers",
    "Loan_Amount_In_Words": "Enter the loan amount in words",
    "Purpose_Of_Loan": "State the purpose for which the loan is being taken",
    "Loan_Period": "Enter the loan duration (e.g., 12 months or 1 year)",
    "Start_Date": "Enter the loan start date (DD/MM/YYYY)",
    "End_Date": "Enter the loan end date (DD/MM/YYYY)",
    "Repayment_Terms": "Enter repayment schedule (e.g., 12 monthly installments)",
    "Repayment_Mode": "Enter repayment method (e.g., bank transfer, UPI)",
    "Interest_Rate": "Enter the interest rate (e.g., 10)",
    "Interest_Basis": "Enter the interest basis (e.g., reducing balance)",
    "Interest_Payment_Frequency": "Enter how often interest is paid (e.g., monthly)",
    "Prepayment_Allowed": "Is prepayment allowed? (Yes/No)",
    "Late_Penalty": "Enter the penalty for late payment (e.g., Rs. 1000/week)",
    "Secured_Unsecured": "Is the loan secured or unsecured?",
    "Security_Details": "Describe any security or collateral (if applicable)",
    "Special_Clauses": "Special terms or clauses if any (auto-filled below)",
    "Jurisdiction": "Enter the legal jurisdiction (e.g., Delhi)",
    "Witness_1_Name": "Enter name of first witness",
    "Witness_1_Signature": "Enter signature initials for witness 1",
    "Witness_2_Name": "Enter name of second witness",
    "Witness_2_Signature": "Enter signature initials for witness 2"
}

# --- Fill Placeholders First ---
print("\n📋 Please answer the following questions to populate the agreement:")
user_inputs = {}
for placeholder, explanation in explanations.items():
    if placeholder == "Special_Clauses":
        continue  # skip; handled later
    value = input(f"🖋 {explanation}: ").strip()
    user_inputs[placeholder] = value
    for para in doc.paragraphs:
        if f"[{placeholder}]" in para.text:
            original_text = para.text
            para.text = para.text.replace(f"[{placeholder}]", value)
            display(Markdown(f"**📄 Updated Line:**\n\n`Before:` {original_text}\n\n`After:` {para.text}"))

# --- Special Clause Prompt Comes AFTER Placeholder Filling ---
special_clause_text = ""
print("\n📄 All standard fields are now filled.")
special_query = input("\n💬 Do you have any special clause to include (e.g., 'part payment allowed')?\n📨 Special Request (leave blank if none): ").strip()
if special_query:
    match_row = find_best_match_semantic(special_query)
    if match_row is not None:
        prompt = build_prompt(match_row['example_1'], match_row['example_2'], match_row['instruction'])
        raw_clause = generate_clause(prompt)
        special_clause_text, _ = fill_parameters_dynamic(raw_clause, match_row.get('parameters', ''))
    else:
        print("⚠️ Could not process your request. Proceeding without it.")

# --- Insert Special Clause in Template ---
if special_clause_text:
    for para in doc.paragraphs:
        if "[Special_Clauses]" in para.text:
            para.text = para.text.replace("[Special_Clauses]", special_clause_text)
            display(Markdown(f"**📄 Inserted Special Clause:** {para.text}"))
            break

# --- Save Final Agreement ---
output_file = "completed_loan_agreement.docx"
doc.save(output_file)
files.download(output_file)
print(f"\n✅ Agreement saved as: {output_file}")


Saving Verified_50_Loan_Clause_Examples_final_fixed.csv to Verified_50_Loan_Clause_Examples_final_fixed (1).csv
📂 Upload your Loan Agreement DOCX template:


Saving loan_agreement.docx to loan_agreement (3).docx

📋 Please answer the following questions to populate the agreement:
🖋 Enter the full name of the person receiving the loan: ramesh Kumar


**📄 Updated Line:**

`Before:` [Borrower_Name]

`After:` ramesh Kumar

**📄 Updated Line:**

`Before:` THIS AGREEMENT is made and entered into at [Location] on this [Day] day of [Month], [Year], BETWEEN [Lender_Name], hereinafter referred to as "the Lender," AND [Borrower_Name], hereinafter referred to as "the Borrower" (collectively referred to as "the Parties"), and shall be binding on their respective heirs, legal representatives, executors, administrators, and assigns.

`After:` THIS AGREEMENT is made and entered into at [Location] on this [Day] day of [Month], [Year], BETWEEN [Lender_Name], hereinafter referred to as "the Lender," AND ramesh Kumar, hereinafter referred to as "the Borrower" (collectively referred to as "the Parties"), and shall be binding on their respective heirs, legal representatives, executors, administrators, and assigns.

**📄 Updated Line:**

`Before:` [Borrower_Name]  

`After:` ramesh Kumar  

...


📄 All standard fields are now filled.

💬 Do you have any special clause to include (e.g., 'part payment allowed')?
📨 Special Request (leave blank if none): No payment allowed in cash
🧾 Please provide value for 'preferred_channels': Bank transfer


**📄 Inserted Special Clause:**    Settlement shall be made through Bank transfer to the Lender’s account.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Agreement saved as: completed_loan_agreement.docx
